In [1]:
import os
import datetime
import pickle
import bz2

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

import h5py

In [2]:
date   = datetime.datetime(2018,12,15)
frange = (14.000e6,14.350e6)

# Load Data

In [4]:
ham_dir   = 'ham_data'
ham_fname = 'rsd{!s}.01.hdf5'.format(date.strftime('%Y-%m-%d'))
ham_fpath = os.path.join(ham_dir,ham_fname)
print(ham_fpath)

ham_data/rsd2018-12-15.01.hdf5


In [5]:
with h5py.File(ham_fpath,'r') as h5_ham:
    data     = h5_ham['Data']
    metadata = h5_ham['Metadata']
    df       = pd.DataFrame(data['Table Layout'][()])

# Rename min and sec columns to be compatible with datetime keywords.
df = df.rename(columns={'min':'minute','sec':'second'})

## Decode Byte Objects to Strings

In [7]:
## DECODE BYTE OBJECTS TO STRINGS
# Select columns that are byte objects
str_df = df.select_dtypes([object])

# Convert all columns to strings
str_df = str_df.stack().str.decode('utf-8').unstack()

# Replace columns in original df
for col in str_df:
    df[col] = str_df[col]

# Delete str_df from memory
del str_df

## Convert Datetimes

In [9]:
# Convert Datetimes
df['date'] = pd.to_datetime(df[['year','month','day','hour','minute','second']])

In [10]:
# Select only Columns of Interest
# for key in df.keys():
#     print(f"keys.append('{key}')")
keys = []
keys.append('date')
keys.append('call_sign_tx')
keys.append('txlat')
keys.append('txlon')
keys.append('call_sign_rx')
keys.append('rxlat')
keys.append('rxlon')
keys.append('tfreq')
keys.append('sn')
keys.append('smode')
keys.append('ssrc')
keys.append('pthlen')
keys.append('latcen')
keys.append('loncen')
df = df[keys]

## Select Frequencies of Interest

In [12]:
tf = np.logical_and(df['tfreq'] >= frange[0], df['tfreq'] < frange[1])
df = df[tf].copy()

In [13]:
df

,date,call_sign_tx,txlat,txlon,call_sign_rx,rxlat,rxlon,tfreq,sn,smode,ssrc,pthlen,latcen,loncen
7,2018-12-15 00:00:00,N8MDP,41.3958,-81.2083,KN3A,40.0625,-76.4583,14075455.0,-18.0,'FT8',PSK,426.8,40.7535,-78.8095
33,2018-12-15 00:00:00,W1JGM,41.4375,-73.5417,N6PAA,38.2313,-121.5792,14075301.0,-18.0,'FT8',PSK,4064.5,42.4027,-98.1565
38,2018-12-15 00:00:00,W0PE,33.8919,-117.7835,N7BT,48.8125,-122.5417,14075574.0,-20.0,'FT8',PSK,1705.1,41.3764,-119.8882
53,2018-12-15 00:00:00,RQ0C,48.5208,135.0417,JK1JAS,35.3542,139.5417,14074632.0,-9.0,'FT8',PSK,1509.9,41.9592,137.5251
54,2018-12-15 00:00:00,RQ0C,48.5208,135.0417,BG8FT,30.5625,103.5417,14074732.0,-16.0,'FT8',PSK,3320.4,40.6026,117.1847
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10101657,2018-12-15 23:59:59,YB1RUS,-6.1875,106.7083,JE2UFF,34.8542,138.1250,14076053.0,-9.0,'FT8',PSK,5627.3,14.8601,120.8759
10101658,2018-12-15 23:59:59,YB1RUS,-6.1875,106.7083,JA1AZR,36.6458,140.7083,14076055.0,-4.0,'FT8',PSK,5931.9,15.8826,121.8385
10101693,2018-12-15 23:59:59,W2HTS,40.2999,-74.0707,W9HM,44.2813,-88.3792,14074407.0,-6.0,'FT8',PSK,1255.4,42.5133,-80.9975
10101775,2018-12-15 23:59:59,LU1XU,-54.8021,-68.3042,LU3XCC,-51.6354,-69.2292,14074983.0,-16.0,'FT8',PSK,357.5,-53.2196,-68.7838


# Output Reduced CSV

In [15]:
out_fname = '{!s}_{!s}-{!s}kHz_hamSpot'.format(
    date.strftime('%Y%m%d'),int(frange[0]/1e3),int(frange[1]/1e3))
out_fpath = os.path.join(ham_dir,out_fname)
print(out_fpath)

ham_data/20181215_14000-14350kHz_hamSpot


In [16]:
df.to_csv(f'{out_fpath}.csv.bz2',index=False)

In [17]:
# Pickle the data
pickled_data = pickle.dumps(df)
# Compress the pickled data
compressed_data = bz2.compress(pickled_data)
# Save the compressed data to a file
with open(f'{out_fpath}.pkl.bz2', "wb") as fl:
    fl.write(compressed_data)